# Notebook 01 — EDA, Pipeline & Preprocessing
**GWHD 2021 | CVProject 2026 Team 13**

This notebook is fully self-contained. All code is written inline here — no `src/` imports needed.

**What this notebook does end-to-end:**
1. Load & verify dataset (CSVs + images)
2. Clean & validate annotations
3. Exploratory Data Analysis (7 plots)
4. Image preprocessing (resize, normalize)
5. Data augmentation (visualized)
6. YOLO label conversion → `.txt` files
7. PyTorch DataLoader smoke test

> After running all cells top-to-bottom, `processed_data/` is ready for training notebooks.

## 0 — Paths, Imports & GPU Check

In [14]:
import os, sys, json, random
import numpy as np
import pandas as pd
import cv2
import matplotlib
matplotlib.use('Agg')   # change to 'inline' if plots don't show
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from tqdm import tqdm
import albumentations as A
from albumentations import BboxParams
import torch
from torch.utils.data import Dataset, DataLoader

# ── Project paths — edit DATA_DIR if your dataset is elsewhere ────────────────
NOTEBOOK_DIR     = os.getcwd()

PROJECT_ROOT     = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))


DATA_DIR         = os.path.join(PROJECT_ROOT + '\gwhd_2021')   # dataset outside project

IMAGES_DIR       = os.path.join(DATA_DIR, 'images')

TRAIN_CSV        = os.path.join(DATA_DIR, 'competition_train.csv')
VAL_CSV          = os.path.join(DATA_DIR, 'competition_val.csv')
TEST_CSV         = os.path.join(DATA_DIR, 'competition_test.csv')
METADATA_CSV     = os.path.join(DATA_DIR, 'metadata_dataset.csv')

PROCESSED_DIR    = os.path.join(PROJECT_ROOT, 'processed_data')
YOLO_LABELS_DIR  = os.path.join(PROCESSED_DIR, 'yolo_labels')
EDA_PLOTS_DIR    = os.path.join(PROJECT_ROOT, 'outputs', 'eda_plots')
SAMPLE_ANN_DIR   = os.path.join(EDA_PLOTS_DIR, 'sample_annotations')

ORIGINAL_IMG_SIZE = 1024
TARGET_IMG_SIZE   = 640
AUGMENTATION_SEED = 42

for d in [PROCESSED_DIR, YOLO_LABELS_DIR, EDA_PLOTS_DIR, SAMPLE_ANN_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Data dir     : {DATA_DIR}')
print(f'Images dir   : {IMAGES_DIR}')
print(f'Images exist : {os.path.isdir(IMAGES_DIR)}')
print(f'PyTorch      : {torch.__version__}')
print(f'CUDA         : {torch.cuda.is_available()}')


Project root : d:\CVProject_2026_Team13_Isometric_v3\CVProject_2026_Team13_Isometric
Data dir     : d:\CVProject_2026_Team13_Isometric_v3\CVProject_2026_Team13_Isometric\gwhd_2021
Images dir   : d:\CVProject_2026_Team13_Isometric_v3\CVProject_2026_Team13_Isometric\gwhd_2021\images
Images exist : True
PyTorch      : 2.10.0+cu128
CUDA         : True


<>:22: SyntaxWarning: invalid escape sequence '\g'
<>:22: SyntaxWarning: invalid escape sequence '\g'
C:\Users\Student\AppData\Local\Temp\ipykernel_21488\1386474702.py:22: SyntaxWarning: invalid escape sequence '\g'
  DATA_DIR         = os.path.join(PROJECT_ROOT + '\gwhd_2021')   # dataset outside project


## Step 1 — Data Loading & Verification

In [15]:
# ── Parse BoxesString ────────────────────────────────────────────────────────
def parse_boxes(box_string):
    """
    Convert 'x1 y1 x2 y2;x1 y1 x2 y2;...' into list of [x1,y1,x2,y2] ints.
    Returns [] for empty or NaN strings.
    """
    if pd.isna(box_string) or str(box_string).strip() == '': return []
    boxes = []
    for box in str(box_string).split(';'):
        parts = box.strip().split()
        if len(parts) == 4:
            boxes.append([int(p) for p in parts])
    return boxes

# ── Load CSVs ────────────────────────────────────────────────────────────────
train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)
meta_df  = pd.read_csv(METADATA_CSV, sep=';')

print(f'Train : {len(train_df):>5} rows')
print(f'Val   : {len(val_df):>5} rows')
print(f'Test  : {len(test_df):>5} rows')
print(f'Meta  : {len(meta_df):>5} rows')
train_df.head(3)


Train :  3657 rows
Val   :  1476 rows
Test  :  1382 rows
Meta  :    47 rows


,image_name,BoxesString,domain
0,4563856cc6d75c670eafd86d5eb7245fbe8f273c28f9e3...,99 692 160 764;641 27 697 115;935 978 1012 102...,Arvalis_1
1,a2a15938845d9812de03bd44799c4b1bf856a8ad11752e...,230 143 321 222;928 929 1015 1004;485 557 604 ...,Arvalis_1
2,401f89a2bb6ab63e3f406bd59b9cadccfe953230feb6cd...,440 239 544 288;333 538 429 594;913 171 963 20...,Arvalis_1


In [16]:
# ── Verify images on disk ────────────────────────────────────────────────────
all_df       = pd.concat([train_df, val_df, test_df], ignore_index=True)
referenced   = set(all_df['image_name'].unique())
on_disk      = set(os.listdir(IMAGES_DIR)) if os.path.isdir(IMAGES_DIR) else set()
missing      = referenced - on_disk
orphan       = on_disk - referenced

print(f'Images referenced in CSVs : {len(referenced)}')
print(f'Images found on disk      : {len(on_disk)}')
print(f'Missing from disk         : {len(missing)}')
print(f'Orphan (unreferenced)     : {len(orphan)}')

# Count total wheat heads
total_heads = sum(len(parse_boxes(s)) for s in all_df['BoxesString'])
print(f'\nTotal images              : {len(all_df)}')
print(f'Total wheat heads (boxes) : {total_heads}')
print(f'Avg heads per image       : {total_heads/len(all_df):.1f}')
print(f'Unique domains            : {all_df["domain"].nunique()}')


Images referenced in CSVs : 6512
Images found on disk      : 6515
Missing from disk         : 0
Orphan (unreferenced)     : 3

Total images              : 6515
Total wheat heads (boxes) : 275468
Avg heads per image       : 42.3
Unique domains            : 47


## Step 2 — Data Cleaning & Validation

In [17]:
def _boxes_to_string(boxes):
    if not boxes: return ''
    return ';'.join(f'{b[0]} {b[1]} {b[2]} {b[3]}' for b in boxes)

def validate_and_clean(df, images_dir=IMAGES_DIR, img_size=ORIGINAL_IMG_SIZE, split_name=''):
    report = dict(missing_values=0, duplicates=0, missing_images=0,
                  invalid_boxes=0, oob_clipped=0,
                  rows_before=len(df), images_affected=set())

    # 1. Drop NaN image_name or domain
    mask = df['image_name'].isna() | df['domain'].isna()
    report['missing_values'] = int(mask.sum())
    df = df[~mask].copy()

    # 2. Drop duplicate rows
    dup = df.duplicated(keep='first')
    report['duplicates'] = int(dup.sum())
    df = df[~dup].copy()

    # 3. Drop rows whose image file is missing
    actual = set(os.listdir(images_dir)) if os.path.isdir(images_dir) else set()
    missing_mask = ~df['image_name'].isin(actual)
    report['missing_images'] = int(missing_mask.sum())
    df = df[~missing_mask].copy()

    # 4. Validate bounding boxes row by row
    cleaned_rows = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc=f'  Cleaning {split_name}', leave=False):
        boxes = parse_boxes(row['BoxesString'])
        valid = []
        for b in boxes:
            x1,y1,x2,y2 = b
            if (x2-x1) <= 0 or (y2-y1) <= 0:
                report['invalid_boxes'] += 1; continue
            clipped = False
            if x1<0: x1,clipped=0,True
            if y1<0: y1,clipped=0,True
            if x2>img_size: x2,clipped=img_size,True
            if y2>img_size: y2,clipped=img_size,True
            if clipped: report['oob_clipped'] += 1
            valid.append([x1,y1,x2,y2])
        r = row.copy()
        r['BoxesString'] = _boxes_to_string(valid)
        cleaned_rows.append(r)

    clean = pd.DataFrame(cleaned_rows).reset_index(drop=True)
    report['rows_after'] = len(clean)
    report['rows_removed'] = report['rows_before'] - report['rows_after']
    return clean, report

train_clean, r_train = validate_and_clean(train_df, split_name='train')
val_clean,   r_val   = validate_and_clean(val_df,   split_name='val')
test_clean,  r_test  = validate_and_clean(test_df,  split_name='test')

for name, r in [('train',r_train),('val',r_val),('test',r_test)]:
    print(f'\n{name}: {r["rows_before"]} → {r["rows_after"]} rows  '
          f'(removed {r["rows_removed"]} | '
          f'invalid_boxes={r["invalid_boxes"]} | oob_clipped={r["oob_clipped"]})')

# Save cleaned CSVs
train_clean.to_csv(os.path.join(PROCESSED_DIR,'train_clean.csv'), index=False)
val_clean.to_csv(  os.path.join(PROCESSED_DIR,'val_clean.csv'),   index=False)
test_clean.to_csv( os.path.join(PROCESSED_DIR,'test_clean.csv'),  index=False)
print('\n[✓] Cleaned CSVs saved to processed_data/')



train: 3657 → 3657 rows  (removed 0 | invalid_boxes=0 | oob_clipped=0)

val: 1476 → 1476 rows  (removed 0 | invalid_boxes=0 | oob_clipped=0)

test: 1382 → 1382 rows  (removed 0 | invalid_boxes=0 | oob_clipped=19)

[✓] Cleaned CSVs saved to processed_data/


## Step 3 — Exploratory Data Analysis

All plots are saved to `outputs/eda_plots/` and displayed inline.

In [18]:
# ── Extract bbox stats from the full dataset ─────────────────────────────────
all_clean = pd.concat([train_clean, val_clean, test_clean], ignore_index=True)

widths, heights, areas, heads_per_image = [], [], [], []
for box_str in all_clean['BoxesString']:
    boxes = parse_boxes(box_str)
    heads_per_image.append(len(boxes))
    for b in boxes:
        w = b[2]-b[0]; h = b[3]-b[1]
        widths.append(w); heights.append(h); areas.append(w*h)

print(f'Total boxes   : {len(widths)}')
print(f'Avg width     : {np.mean(widths):.1f} px')
print(f'Avg height    : {np.mean(heights):.1f} px')
print(f'Avg area      : {np.mean(areas):.1f} px²')
print(f'Avg heads/img : {np.mean(heads_per_image):.1f}')
print(f'Max heads/img : {max(heads_per_image)}')


Total boxes   : 275468
Avg width     : 76.6 px
Avg height    : 72.5 px
Avg area      : 6160.4 px²
Avg heads/img : 42.3
Max heads/img : 190


In [19]:
# ── Plot 1: Bounding box size distributions ───────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(widths,  bins=50, color='steelblue', edgecolor='black')
axes[0].set_title('Bbox Width Distribution'); axes[0].set_xlabel('Width (px)')
axes[1].hist(heights, bins=50, color='salmon',   edgecolor='black')
axes[1].set_title('Bbox Height Distribution'); axes[1].set_xlabel('Height (px)')
axes[2].hist(areas,   bins=50, color='mediumseagreen', edgecolor='black')
axes[2].set_title('Bbox Area Distribution'); axes[2].set_xlabel('Area (px²)')
for ax in axes: ax.set_ylabel('Count'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(EDA_PLOTS_DIR,'bbox_statistics.png'), dpi=150)
plt.show(); print('[✓] bbox_statistics.png saved')


[✓] bbox_statistics.png saved


C:\Users\Student\AppData\Local\Temp\ipykernel_21488\467581614.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); print('[✓] bbox_statistics.png saved')


In [20]:
# ── Plot 2: Wheat heads per image histogram ───────────────────────────────────
plt.figure(figsize=(8, 4))
plt.hist(heads_per_image, bins=40, color='darkorange', edgecolor='black')
plt.title('Wheat Heads per Image'); plt.xlabel('Heads'); plt.ylabel('Images')
plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(EDA_PLOTS_DIR,'heads_per_image.png'), dpi=150)
plt.show(); print('[✓] heads_per_image.png saved')


[✓] heads_per_image.png saved


C:\Users\Student\AppData\Local\Temp\ipykernel_21488\1585926413.py:7: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); print('[✓] heads_per_image.png saved')


In [21]:
# ── Plot 3: Images per country ────────────────────────────────────────────────
domain_counts = all_clean['domain'].value_counts().reset_index()
domain_counts.columns = ['domain','image_count']
merged = domain_counts.merge(meta_df[['name','country']],
                              left_on='domain', right_on='name', how='left')
country_counts = merged.groupby('country')['image_count'].sum().sort_values(ascending=False)

plt.figure(figsize=(10,4))
country_counts.plot(kind='bar', color='teal', edgecolor='black')
plt.title('Images per Country'); plt.xlabel('Country'); plt.ylabel('Count')
plt.xticks(rotation=45, ha='right'); plt.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(EDA_PLOTS_DIR,'images_per_country.png'), dpi=150)
plt.show(); print('[✓] images_per_country.png saved')


[✓] images_per_country.png saved


C:\Users\Student\AppData\Local\Temp\ipykernel_21488\2190768959.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); print('[✓] images_per_country.png saved')


In [22]:
# ── Plot 4: Top-15 domains ────────────────────────────────────────────────────
top15 = domain_counts.head(15)
plt.figure(figsize=(12, 4))
sns.barplot(data=top15, x='domain', y='image_count', hue='domain',
            palette='viridis', legend=False)
plt.title('Top-15 Domains by Image Count'); plt.xlabel('Domain'); plt.ylabel('Count')
plt.xticks(rotation=45, ha='right'); plt.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(EDA_PLOTS_DIR,'top15_domains.png'), dpi=150)
plt.show(); print('[✓] top15_domains.png saved')


[✓] top15_domains.png saved


C:\Users\Student\AppData\Local\Temp\ipykernel_21488\1805809201.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); print('[✓] top15_domains.png saved')


In [23]:
# ── Plot 5: Development stage distribution ────────────────────────────────────
merged2 = domain_counts.merge(meta_df[['name','development_stage']],
                               left_on='domain', right_on='name', how='left')
merged2['stage_clean'] = merged2['development_stage'].str.strip().str.lower()
stage_counts = merged2.groupby('stage_clean')['image_count'].sum().sort_values(ascending=False)

plt.figure(figsize=(8,4))
stage_counts.plot(kind='bar', color='mediumpurple', edgecolor='black')
plt.title('Images by Development Stage'); plt.xlabel('Stage'); plt.ylabel('Count')
plt.xticks(rotation=45, ha='right'); plt.grid(axis='y', alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(EDA_PLOTS_DIR,'development_stage.png'), dpi=150)
plt.show(); print('[✓] development_stage.png saved')


[✓] development_stage.png saved


C:\Users\Student\AppData\Local\Temp\ipykernel_21488\907537371.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); print('[✓] development_stage.png saved')


In [24]:
# ── Plot 6: Bbox center heatmap (may take ~30s on full set) ──────────────────
heatmap = np.zeros((ORIGINAL_IMG_SIZE, ORIGINAL_IMG_SIZE), dtype=np.float64)
for box_str in tqdm(all_clean['BoxesString'], desc='Building heatmap', leave=False):
    for b in parse_boxes(box_str):
        cx = min((b[0]+b[2])//2, ORIGINAL_IMG_SIZE-1)
        cy = min((b[1]+b[3])//2, ORIGINAL_IMG_SIZE-1)
        heatmap[cy, cx] += 1
heatmap = cv2.GaussianBlur(heatmap, (51,51), 0)

plt.figure(figsize=(7,7))
plt.imshow(heatmap, cmap='hot'); plt.colorbar(label='Density')
plt.title('Bbox Center Heatmap'); plt.xlabel('X'); plt.ylabel('Y')
plt.tight_layout()
plt.savefig(os.path.join(EDA_PLOTS_DIR,'bbox_center_heatmap.png'), dpi=150)
plt.show(); print('[✓] bbox_center_heatmap.png saved')


[✓] bbox_center_heatmap.png saved


C:\Users\Student\AppData\Local\Temp\ipykernel_21488\693365528.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show(); print('[✓] bbox_center_heatmap.png saved')


In [25]:
# ── Plot 7: Sample annotated images ──────────────────────────────────────────
random.seed(42)
df_with_boxes = all_clean[all_clean['BoxesString'].apply(lambda s: len(parse_boxes(s))>0)]
sample_rows   = df_with_boxes.sample(n=min(20, len(df_with_boxes)), random_state=42)

saved = 0
for _, row in sample_rows.iterrows():
    img_path = os.path.join(IMAGES_DIR, row['image_name'])
    img = cv2.imread(img_path)
    if img is None: continue
    for b in parse_boxes(row['BoxesString']):
        cv2.rectangle(img, (b[0],b[1]), (b[2],b[3]), (0,255,0), 2)
    cv2.imwrite(os.path.join(SAMPLE_ANN_DIR, row['image_name']), img)
    saved += 1
print(f'[✓] {saved} annotated samples saved to {SAMPLE_ANN_DIR}')

# Show 4 inline
samples = [f for f in os.listdir(SAMPLE_ANN_DIR) if f.endswith('.png')][:4]
fig, axes = plt.subplots(1, len(samples), figsize=(20, 5))
for ax, fname in zip(axes, samples):
    img = cv2.cvtColor(cv2.imread(os.path.join(SAMPLE_ANN_DIR,fname)), cv2.COLOR_BGR2RGB)
    ax.imshow(img); ax.axis('off'); ax.set_title(fname[:24], fontsize=8)
plt.suptitle('Sample Annotated Images', fontsize=12); plt.tight_layout(); plt.show()


[✓] 20 annotated samples saved to d:\CVProject_2026_Team13_Isometric_v3\CVProject_2026_Team13_Isometric\outputs\eda_plots\sample_annotations


C:\Users\Student\AppData\Local\Temp\ipykernel_21488\2512295343.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.suptitle('Sample Annotated Images', fontsize=12); plt.tight_layout(); plt.show()


## Step 4 — Image Preprocessing

Shows resize + normalize on a real image. Resizing is applied on-the-fly during training
(inside the Dataset class), so we do NOT re-save all 3000+ images to disk here.

In [26]:
# ── Resize helper ────────────────────────────────────────────────────────────
def resize_image(image, target_size=TARGET_IMG_SIZE):
    """Resize image to (target_size, target_size) using bilinear interpolation."""
    return cv2.resize(image, (target_size, target_size), interpolation=cv2.INTER_LINEAR)

def rescale_boxes(boxes, orig_size=ORIGINAL_IMG_SIZE, target_size=TARGET_IMG_SIZE):
    """Scale bounding boxes proportionally from orig_size to target_size."""
    scale = target_size / orig_size
    return [[int(b[0]*scale), int(b[1]*scale),
             int(b[2]*scale), int(b[3]*scale)] for b in boxes]

def normalize_image(image):
    """Scale pixel values [0,255] → [0.0,1.0] float32."""
    return image.astype(np.float32) / 255.0

# ── Demo on one image ─────────────────────────────────────────────────────────
sample_row  = train_clean.iloc[0]
img_path    = os.path.join(IMAGES_DIR, sample_row['image_name'])
orig_img    = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
boxes_orig  = parse_boxes(sample_row['BoxesString'])

resized_img   = resize_image(orig_img)
boxes_resized = rescale_boxes(boxes_orig)
norm_img      = normalize_image(resized_img)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(orig_img)
for b in boxes_orig[:10]:  # show first 10 boxes
    axes[0].add_patch(mpatches.Rectangle((b[0],b[1]),b[2]-b[0],b[3]-b[1],
                      linewidth=1.5, edgecolor='lime', facecolor='none'))
axes[0].set_title(f'Original  {orig_img.shape[1]}×{orig_img.shape[0]}px', fontsize=11)
axes[0].axis('off')

axes[1].imshow(resized_img)
for b in boxes_resized[:10]:
    axes[1].add_patch(mpatches.Rectangle((b[0],b[1]),b[2]-b[0],b[3]-b[1],
                      linewidth=1.5, edgecolor='lime', facecolor='none'))
axes[1].set_title(f'Resized  {TARGET_IMG_SIZE}×{TARGET_IMG_SIZE}px  |  boxes rescaled', fontsize=11)
axes[1].axis('off')

plt.suptitle(f'Preprocessing demo — {sample_row["image_name"]}', fontsize=11)
plt.tight_layout(); plt.show()
print(f'Pixel range before normalize: [{orig_img.min()}, {orig_img.max()}]')
print(f'Pixel range after normalize : [{norm_img.min():.2f}, {norm_img.max():.2f}]')


Pixel range before normalize: [0, 255]
Pixel range after normalize : [0.00, 1.00]


C:\Users\Student\AppData\Local\Temp\ipykernel_21488\1154593069.py:42: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Step 5 — Data Augmentation

Augmentations are applied on-the-fly during training. Here we visualize what each transform does.

In [27]:
# ── Build Albumentations pipeline ────────────────────────────────────────────
def get_augmentation_pipeline(target_size=TARGET_IMG_SIZE):
    return A.Compose([
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.GaussianBlur(blur_limit=(3,7), p=0.3),
        A.Affine(
            translate_percent={'x':(-0.05,0.05), 'y':(-0.05,0.05)},
            scale=(0.9,1.1), rotate=(-15,15),
            border_mode=cv2.BORDER_CONSTANT, p=0.5
        ),
    ], bbox_params=BboxParams(
        format='pascal_voc',
        label_fields=['class_labels'],
        min_visibility=0.3,
    ))

def apply_augmentation(image, boxes, transform=None):
    if transform is None: transform = get_augmentation_pipeline()
    if len(boxes) == 0:
        return transform(image=image, bboxes=[], class_labels=[])['image'], []
    result  = transform(image=image, bboxes=boxes, class_labels=[0]*len(boxes))
    return result['image'], [list(b) for b in result['bboxes']]

aug_transform = get_augmentation_pipeline()
print('[✓] Augmentation pipeline ready')
print('Transforms: HorizontalFlip | VerticalFlip | BrightnessContrast | GaussianBlur | Affine')


[✓] Augmentation pipeline ready
Transforms: HorizontalFlip | VerticalFlip | BrightnessContrast | GaussianBlur | Affine


In [28]:
# ── Visualize: original vs 4 augmented versions ───────────────────────────────
demo_row   = train_clean.sample(1, random_state=7).iloc[0]
demo_path  = os.path.join(IMAGES_DIR, demo_row['image_name'])
demo_img   = cv2.cvtColor(cv2.imread(demo_path), cv2.COLOR_BGR2RGB)
demo_img   = resize_image(demo_img)
demo_boxes = rescale_boxes(parse_boxes(demo_row['BoxesString']))

fig, axes = plt.subplots(1, 5, figsize=(24, 5))

def draw_boxes(ax, img, boxes, title):
    ax.imshow(img)
    for b in boxes:
        ax.add_patch(mpatches.Rectangle((b[0],b[1]),b[2]-b[0],b[3]-b[1],
                     linewidth=1.2, edgecolor='lime', facecolor='none'))
    ax.set_title(f'{title}\n{len(boxes)} boxes', fontsize=9); ax.axis('off')

draw_boxes(axes[0], demo_img, demo_boxes, 'Original')
for i in range(1, 5):
    aug_img, aug_boxes = apply_augmentation(demo_img.copy(), [list(b) for b in demo_boxes])
    draw_boxes(axes[i], aug_img, aug_boxes, f'Aug #{i}')

plt.suptitle('Augmentation Visualization', fontsize=12)
plt.tight_layout(); plt.show()
print('[✓] Augmentation visualization done')


[✓] Augmentation visualization done


C:\Users\Student\AppData\Local\Temp\ipykernel_21488\3932896269.py:23: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Step 6 — YOLO Label Conversion

Converts `[x1,y1,x2,y2]` pixel boxes → YOLO format: `class_id cx cy w h` (all normalised to [0,1]).
Saves one `.txt` file per image in `processed_data/yolo_labels/`.

In [29]:
def boxes_to_yolo(boxes, img_size=TARGET_IMG_SIZE):
    """Convert list of [x1,y1,x2,y2] to YOLO normalised format strings."""
    lines = []
    for (x1,y1,x2,y2) in boxes:
        cx = ((x1+x2)/2) / img_size
        cy = ((y1+y2)/2) / img_size
        w  = (x2-x1)    / img_size
        h  = (y2-y1)    / img_size
        lines.append(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    return lines

def save_yolo_labels(df, save_dir=YOLO_LABELS_DIR,
                     orig_size=ORIGINAL_IMG_SIZE, target_size=TARGET_IMG_SIZE):
    os.makedirs(save_dir, exist_ok=True)
    for _, row in tqdm(df.iterrows(), total=len(df), desc='Writing YOLO labels', leave=False):
        boxes = parse_boxes(row['BoxesString'])
        boxes = rescale_boxes(boxes, orig_size, target_size)
        lines = boxes_to_yolo(boxes, target_size)
        stem  = os.path.splitext(row['image_name'])[0]
        with open(os.path.join(save_dir, f'{stem}.txt'), 'w') as f:
            f.write('\n'.join(lines))

# Run for all splits
all_clean_combined = pd.concat([train_clean, val_clean, test_clean], ignore_index=True)
save_yolo_labels(all_clean_combined)

n_labels = len(os.listdir(YOLO_LABELS_DIR))
print(f'[✓] {n_labels} YOLO label files saved to {YOLO_LABELS_DIR}')


[✓] 6512 YOLO label files saved to d:\CVProject_2026_Team13_Isometric_v3\CVProject_2026_Team13_Isometric\processed_data\yolo_labels


In [30]:
# ── Verify: show a label file alongside its image ─────────────────────────────
sample_stem  = os.path.splitext(train_clean.iloc[0]['image_name'])[0]
label_path   = os.path.join(YOLO_LABELS_DIR, f'{sample_stem}.txt')

with open(label_path) as f:
    label_lines = f.readlines()

print(f'Label file : {label_path}')
print(f'Boxes      : {len(label_lines)}')
print('Format     : class_id  cx  cy  w  h  (all normalised to [0,1])')
print('First 5 lines:')
for l in label_lines[:5]: print(f'  {l.strip()}')

# Draw YOLO boxes on resized image to verify correctness
check_img  = resize_image(cv2.cvtColor(cv2.imread(os.path.join(IMAGES_DIR, train_clean.iloc[0]['image_name'])), cv2.COLOR_BGR2RGB))
fig, ax    = plt.subplots(figsize=(7,7))
ax.imshow(check_img)
S = TARGET_IMG_SIZE
for line in label_lines[:30]:
    _, cx, cy, w, h = map(float, line.strip().split())
    x1 = (cx - w/2)*S; y1 = (cy - h/2)*S
    ax.add_patch(mpatches.Rectangle((x1,y1), w*S, h*S,
                 linewidth=1.2, edgecolor='red', facecolor='none'))
ax.set_title(f'YOLO label verification — {len(label_lines)} boxes', fontsize=11)
ax.axis('off'); plt.tight_layout(); plt.show()


Label file : d:\CVProject_2026_Team13_Isometric_v3\CVProject_2026_Team13_Isometric\processed_data\yolo_labels\4563856cc6d75c670eafd86d5eb7245fbe8f273c28f9e36f7c6aaf097c7ce423.txt
Boxes      : 41
Format     : class_id  cx  cy  w  h  (all normalised to [0,1])
First 5 lines:
  0 0.125781 0.710156 0.060937 0.070312
  0 0.652344 0.067969 0.054688 0.085938
  0 0.950000 0.975000 0.075000 0.040625
  0 0.409375 0.841406 0.084375 0.054688
  0 0.658594 0.797656 0.042188 0.039062


C:\Users\Student\AppData\Local\Temp\ipykernel_21488\2356390123.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.axis('off'); plt.tight_layout(); plt.show()


## Step 7 — PyTorch Dataset & DataLoader Smoke Test

The `WheatDataset` class applies resize + augmentation on-the-fly so we never duplicate the images on disk.

In [31]:
class WheatDataset(Dataset):
    """
    PyTorch Dataset for GWHD 2021.
    Loads image, resizes, optionally augments, normalizes, returns tensor + target.
    """
    def __init__(self, df, images_dir=IMAGES_DIR,
                 target_size=TARGET_IMG_SIZE, augment=False):
        self.df         = df.reset_index(drop=True)
        self.images_dir = images_dir
        self.target_size= target_size
        self.augment    = augment
        self.transform  = get_augmentation_pipeline(target_size) if augment else None

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = cv2.imread(os.path.join(self.images_dir, row['image_name']))
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img  = resize_image(img, self.target_size)
        boxes= rescale_boxes(parse_boxes(row['BoxesString']),
                              ORIGINAL_IMG_SIZE, self.target_size)

        if self.augment and self.transform and len(boxes) > 0:
            img, boxes = apply_augmentation(img, boxes, self.transform)

        img  = normalize_image(img)
        img_t= torch.from_numpy(img).permute(2,0,1).float()  # (3,H,W)

        boxes_t  = torch.tensor(boxes, dtype=torch.float32) if boxes else torch.zeros((0,4))
        labels_t = torch.zeros(len(boxes), dtype=torch.long)
        return img_t, {'boxes': boxes_t, 'labels': labels_t}

def collate_fn(batch):
    images, targets = zip(*batch)
    return torch.stack(images), list(targets)

print('[✓] WheatDataset and collate_fn defined')


[✓] WheatDataset and collate_fn defined


In [32]:
# ── Build loaders ────────────────────────────────────────────────────────────
train_ds = WheatDataset(train_clean, augment=True)
val_ds   = WheatDataset(val_clean,   augment=False)

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True,
                          num_workers=0, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=4, shuffle=False,
                          num_workers=0, collate_fn=collate_fn)

# Fetch one batch and verify shapes
imgs, targets = next(iter(train_loader))
print(f'Batch image tensor : {imgs.shape}')           # expect (4, 3, 640, 640)
print(f'Pixel range        : [{imgs.min():.2f}, {imgs.max():.2f}]')  # expect [0,1]
print(f'Targets in batch   : {len(targets)}')
print(f'First img boxes    : {targets[0]["boxes"].shape}')
print(f'First img labels   : {targets[0]["labels"]}')


Batch image tensor : torch.Size([4, 3, 640, 640])
Pixel range        : [0.00, 1.00]
Targets in batch   : 4
First img boxes    : torch.Size([49, 4])
First img labels   : tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0])


In [33]:
# ── Visualize one augmented training batch ────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
for i, ax in enumerate(axes):
    img_np  = (imgs[i].permute(1,2,0).numpy()*255).astype('uint8')
    boxes_b = targets[i]['boxes'].numpy()
    ax.imshow(img_np)
    for (x1,y1,x2,y2) in boxes_b:
        ax.add_patch(mpatches.Rectangle((x1,y1),x2-x1,y2-y1,
                     linewidth=1.2, edgecolor='lime', facecolor='none'))
    ax.set_title(f'Sample {i+1} — {len(boxes_b)} heads', fontsize=9)
    ax.axis('off')
plt.suptitle('Training batch — augmented & normalized', fontsize=11)
plt.tight_layout(); plt.show()


C:\Users\Student\AppData\Local\Temp\ipykernel_21488\2664109616.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## Summary

| Step | What happened | Output |
|---|---|---|
| 1. Load | CSVs loaded, images verified | `train_df`, `val_df`, `test_df` |
| 2. Clean | Nulls, dupes, bad boxes removed | `processed_data/*.csv` |
| 3. EDA | 7 plots generated | `outputs/eda_plots/` |
| 4. Preprocess | Resize + normalize demonstrated | on-the-fly in Dataset |
| 5. Augment | Pipeline built + visualized | on-the-fly in Dataset |
| 6. YOLO labels | All images converted | `processed_data/yolo_labels/` |
| 7. DataLoader | Batches verified, shapes correct | ready for training |

**Next:** Open `02_yolov8_training.ipynb`.